In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

In [13]:
df = pd.read_csv('ml_no_show_features.csv', sep=";")
df.head()

,age,gender,SMS_received,lead_time_days,appointment_day,appointment_month,age_group,lead_time_group,no_show_binary
0,3,F,0,3,Wednesday,October,Child,30+ Days,0
1,94,M,1,21,Saturday,April,Senior,30+ Days,1
2,29,M,0,6,Thursday,April,Young Adult,30+ Days,0
3,53,M,1,17,Thursday,December,Adult,30+ Days,0
4,20,F,1,24,Saturday,January,Young Adult,30+ Days,0


In [16]:
df.columns = df.columns.str.strip().str.lower()

In [19]:
df_encoded = pd.get_dummies(
    df,
    columns=["gender","appointment_day","appointment_month","age_group","lead_time_group"],
    drop_first=True,
    dtype=int
)

In [20]:
df_encoded.head()

,age,sms_received,lead_time_days,no_show_binary,gender_M,appointment_day_Monday,appointment_day_Saturday,appointment_day_Sunday,appointment_day_Thursday,appointment_day_Tuesday,...,appointment_month_March,appointment_month_May,appointment_month_November,appointment_month_October,appointment_month_September,age_group_Child,age_group_Senior,age_group_Teen,age_group_Young Adult,lead_time_group_Same Day
0,3,0,3,0,0,0,0,0,0,0,...,0,0,0,1,0,1,0,0,0,0
1,94,1,21,1,1,0,1,0,0,0,...,0,0,0,0,0,0,1,0,0,0
2,29,0,6,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,1,0
3,53,1,17,0,1,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,20,1,24,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [21]:
x = df_encoded.drop("no_show_binary", axis=1)
y = df_encoded["no_show_binary"]

In [22]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

In [24]:
model = LogisticRegression(max_iter=1000)
model.fit(x_train,y_train)

LogisticRegression(max_iter=1000)

In [29]:
y_prediction = model.predict(x_test)
y_prob = model.predict_proba(x_test)[:,1]

print("Accuracy:",accuracy_score(y_test, y_prediction))
print("ROC AUC:", roc_auc_score(y_test, y_prob))
print(classification_report(y_test, y_prediction))

Accuracy: 0.794
ROC AUC: 0.4969736616859456
              precision    recall  f1-score   support

           0       0.79      1.00      0.89       794
           1       0.00      0.00      0.00       206

    accuracy                           0.79      1000
   macro avg       0.40      0.50      0.44      1000
weighted avg       0.63      0.79      0.70      1000



C:\Users\Vukani\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Vukani\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Vukani\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [32]:
feature_importance = (
    pd.Series(model.coef_[0], index=x.columns)
    .sort_values(ascending=False)
)

feature_importance.head(15)

lead_time_group_Same Day       0.409152
appointment_day_Sunday         0.232111
appointment_day_Wednesday      0.213375
appointment_day_Monday         0.180203
appointment_day_Tuesday        0.139571
age_group_Young Adult          0.092722
appointment_month_September    0.087189
appointment_day_Saturday       0.053038
appointment_day_Thursday       0.053031
appointment_month_October      0.038196
appointment_month_June         0.022328
gender_M                       0.021729
lead_time_days                 0.010686
sms_received                   0.008508
age                            0.000252
dtype: float64

In [33]:
df_encoded["no_show_risk_score"] = model.predict_proba(x)[:,1]
df_encoded[["no_show_risk_score"]].head()

,no_show_risk_score
0,0.164176
1,0.219366
2,0.219109
3,0.213738
4,0.190400
